In [1]:
import pandas as pd



In [2]:
census = pd.read_csv("../data/external/census_indicators.csv")
print(census.shape)
print(census.columns.tolist())
census.tail(3)

(78, 9)
['Community Area Number', 'COMMUNITY AREA NAME', 'PERCENT OF HOUSING CROWDED', 'PERCENT HOUSEHOLDS BELOW POVERTY', 'PERCENT AGED 16+ UNEMPLOYED', 'PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA', 'PERCENT AGED UNDER 18 OR OVER 64', 'PER CAPITA INCOME ', 'HARDSHIP INDEX']


,Community Area Number,COMMUNITY AREA NAME,PERCENT OF HOUSING CROWDED,PERCENT HOUSEHOLDS BELOW POVERTY,PERCENT AGED 16+ UNEMPLOYED,PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA,PERCENT AGED UNDER 18 OR OVER 64,PER CAPITA INCOME,HARDSHIP INDEX
75,76.0,O'Hare,3.6,15.4,7.1,10.9,30.3,25828,24.0
76,77.0,Edgewater,4.1,18.2,9.2,9.7,23.8,33385,19.0
77,NaN,CHICAGO,4.7,19.7,12.9,19.5,33.5,28202,NaN


In [3]:
census = census.rename(columns={
    "Community Area Number": "community_area",
    "COMMUNITY AREA NAME": "area_name",
    "PERCENT OF HOUSING CROWDED": "pct_crowded",
    "PERCENT HOUSEHOLDS BELOW POVERTY": "poverty_rate",
    "PERCENT AGED 16+ UNEMPLOYED": "unemployment",
    "PERCENT AGED 25+ WITHOUT HIGH SCHOOL DIPLOMA": "pct_no_diploma",
    "PERCENT AGED UNDER 18 OR OVER 64": "pct_dependent",
    "PER CAPITA INCOME": "per_capita_income",
    "HARDSHIP INDEX": "hardship_index",
})

# drop the whole-city row and make community_area an integer
census = census.dropna(subset=["community_area"])
census["community_area"] = census["community_area"].astype(int)

print(census.shape)                       # should be (77, 9)
print(census["community_area"].nunique()) # should be 77
census.to_parquet("../data/processed/census_clean.parquet", index=False)

(77, 9)
77


In [5]:
pop = pd.read_csv("../data/external/population.csv")
print(pop.shape)
print(pop.columns.tolist())
pop.head()

(235, 26)
['Geography Type', 'Year', 'Geography', 'Population - Total', 'Population - Age 0-17', 'Population - Age 18-29', 'Population - Age 30-39', 'Population - Age 40-49', 'Population - Age 50-59', 'Population - Age 60-69', 'Population - Age 70-79', 'Population - Age 80+', 'Population - Age 0-4', 'Population - Age 5-11', 'Population - Age 12-17', 'Population - Age 5+', 'Population - Age 18+', 'Population - Age 65+', 'Population - Female', 'Population - Male', 'Population - Latinx', 'Population - Asian Non-Latinx', 'Population - Black Non-Latinx', 'Population - White Non-Latinx', 'Population - Other Race Non-Latinx', 'Record ID']


,Geography Type,Year,Geography,Population - Total,Population - Age 0-17,Population - Age 18-29,Population - Age 30-39,Population - Age 40-49,Population - Age 50-59,Population - Age 60-69,...,Population - Age 18+,Population - Age 65+,Population - Female,Population - Male,Population - Latinx,Population - Asian Non-Latinx,Population - Black Non-Latinx,Population - White Non-Latinx,Population - Other Race Non-Latinx,Record ID
0,Citywide,2018,Chicago,"2,705,988","548,999","552,935","456,321","336,457","312,965","262,991",...,"2,156,989","349,712","1,386,113","1,319,875","776,661","179,841","784,266","899,980","119,467",Citywide-Chicago-2018
1,ZIP Code,2018,60601,"14,675",820,"4,606","2,792","2,190","1,333","1,340",...,"13,855","2,075","7,484","7,191","1,274",NaN,NaN,"9,677",NaN,ZIP_Code-60601-2018
2,ZIP Code,2018,60602,"1,244",149,435,462,135,53,10,...,"1,095",5,551,693,81,NaN,NaN,788,NaN,ZIP_Code-60602-2018
3,ZIP Code,2018,60603,"1,174",56,561,101,97,197,97,...,"1,118",112,601,573,115,NaN,NaN,707,NaN,ZIP_Code-60603-2018
4,ZIP Code,2018,60604,782,38,303,104,51,101,130,...,744,93,413,369,34,NaN,NaN,479,NaN,ZIP_Code-60604-2018


In [6]:
print(pop["Geography Type"].value_counts())
print(pop["Year"].unique())


Geography Type
ZIP Code    231
Citywide      4
Name: count, dtype: int64
[2018 2019 2020 2021]


In [7]:
import os
print(os.listdir("../data/external"))   # check the exact file name

cca = pd.read_csv("../data/external/cmap_cca.csv", low_memory=False)   # change the name if needed
print(cca.shape)                         # expect 77 rows
print(cca.columns.tolist()[:15])         # first 15 columns
cca[["GEOID", "GEOG", "F2020_POP"]].head(10)


['census_indicators.csv', 'cmap_cca.csv', 'population.csv']
(77, 181)
['fid', 'GEOID', 'GEOG', 'F2000_POP', 'F2010_POP', 'F2020_POP', 'F2020_HH', 'TOT_POP', 'UND5', 'A5_19', 'A20_34', 'A35_49', 'A50_64', 'A65_74', 'A75_84']


,GEOID,GEOG,F2020_POP
0,1,Rogers Park,55628
1,10,Norwood Park,40292
2,11,Jefferson Park,26216
3,12,Forest Glen,19596
4,13,North Park,17559
5,14,Albany Park,48396
6,15,Portage Park,64040
7,16,Irving Park,51940
8,17,Dunning,43147
9,18,Montclare,14401


In [8]:
pop = cca[["GEOID", "GEOG", "F2020_POP"]].copy()
pop.columns = ["community_area", "cmap_name", "population"]
pop["community_area"] = pop["community_area"].astype(int)
pop["population"] = pd.to_numeric(pop["population"], errors="coerce")

print(pop.shape)                                          # (77, 3)
print("Missing populations:", pop["population"].isna().sum())   # 0
print("Total population:", pop["population"].sum())             # about 2.7 million
print("Unique areas:", pop["community_area"].nunique())         # 77

(77, 3)
Missing populations: 0
Total population: 2784433
Unique areas: 77


In [9]:
area_info = census.merge(pop[["community_area", "population"]],
                         on="community_area", how="left")

print(area_info.shape)                                    # (77, 10)
print("Missing population:", area_info["population"].isna().sum())   # 0
area_info.head()

(77, 10)
Missing population: 0


,community_area,area_name,pct_crowded,poverty_rate,unemployment,pct_no_diploma,pct_dependent,PER CAPITA INCOME,hardship_index,population
0,1,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0,55628
1,2,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0,77122
2,3,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0,57182
3,4,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0,40494
4,5,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0,35114


In [10]:
check = census[["community_area", "area_name"]].merge(
    pop[["community_area", "cmap_name"]], on="community_area")
check["match"] = check["area_name"].str.upper().str.strip() == check["cmap_name"].str.upper().str.strip()
print("Names matching:", check["match"].sum(), "of 77")
print(check[~check["match"]])

Names matching: 74 of 77
    community_area          area_name           cmap_name  match
17              18         Montclaire           Montclare  False
31              32               Loop            The Loop  False
72              73  Washington Height  Washington Heights  False


In [11]:
area_info.to_parquet("../data/processed/area_info.parquet", index=False)
print("Saved", area_info.shape)

Saved (77, 10)


In [12]:
# rename whatever the income column is actually called
income_col = [c for c in area_info.columns if "INCOME" in c.upper()][0]
area_info = area_info.rename(columns={income_col: "per_capita_income"})
print(area_info.columns.tolist())

['community_area', 'area_name', 'pct_crowded', 'poverty_rate', 'unemployment', 'pct_no_diploma', 'pct_dependent', 'per_capita_income', 'hardship_index', 'population']


In [13]:
area_info["area_name"] = area_info["area_name"].replace({
    "Montclaire": "Montclare",
    "Loop": "The Loop",
    "Washington Height": "Washington Heights",
})
print(area_info.loc[area_info["community_area"].isin([18, 32, 73]), ["community_area", "area_name"]])

    community_area           area_name
17              18           Montclare
31              32            The Loop
72              73  Washington Heights


In [14]:
area_info.to_parquet("../data/processed/area_info.parquet", index=False)
print(area_info.shape)          # (77, 10)
print(area_info.isna().sum().sum(), "missing values in total")   # expect 0


(77, 10)
0 missing values in total


In [15]:
import importlib.metadata as md
print("meteostat:", md.version("meteostat"))
print("holidays:", md.version("holidays"))

meteostat: 2.1.5
holidays: 0.106


In [16]:
from datetime import date
import meteostat as ms

POINT = ms.Point(41.8781, -87.6298, 181)       # Chicago (181 m elevation)
START = date(2023, 1, 1)
END = date(2026, 9, 10)                        # same end date as our cleaned crime data

stations = ms.stations.nearby(POINT, limit=4)  # 4 closest weather stations
ts = ms.daily(stations, START, END)
weather = ms.interpolate(ts, POINT).fetch()    # one blended value per day

print(weather.shape)
print(weather.columns.tolist())
print(weather.index.min(), "->", weather.index.max())
print(weather.isna().sum())
weather.head()

(1349, 11)
['temp', 'tmin', 'tmax', 'rhum', 'prcp', 'snwd', 'wspd', 'wpgt', 'pres', 'tsun', 'cldc']
2023-01-01 00:00:00 -> 2026-09-10 00:00:00
temp       0
tmin       0
tmax       0
rhum       0
prcp       0
snwd       0
wspd       0
wpgt    1343
pres       0
tsun    1349
cldc       0
dtype: int64


,temp,tmin,tmax,rhum,prcp,snwd,wspd,wpgt,pres,tsun,cldc
time,,,,,,,,,,,
2023-01-01,5.1,2.7,7.4,90,3.6,0,8.1,<NA>,1014.1,<NA>,7
2023-01-02,3.5,0.2,6.4,90,0.0,0,9.1,<NA>,1017.1,<NA>,8
2023-01-03,6.2,4.3,9.7,96,27.3,0,10.2,<NA>,1004.2,<NA>,8
2023-01-04,3.0,0.6,7.6,83,0.2,0,13.8,<NA>,1007.0,<NA>,7
2023-01-05,0.5,-0.4,1.2,82,0.7,0,15.6,<NA>,1012.4,<NA>,8


In [17]:
weather_daily = weather[["temp", "prcp"]].copy()
weather_daily.columns = ["temperature", "rainfall"]
weather_daily["temperature"] = weather_daily["temperature"].astype(float)
weather_daily["rainfall"] = weather_daily["rainfall"].astype(float)
weather_daily = weather_daily.reset_index().rename(columns={"time": "date"})
weather_daily["date"] = pd.to_datetime(weather_daily["date"]).dt.normalize()

print(weather_daily.shape)                              # (1349, 3)
print(weather_daily.isna().sum().tolist(), "missing")   # [0, 0, 0]
print(weather_daily.describe().round(1))

weather_daily.to_parquet("../data/processed/weather_daily.parquet", index=False)
print("Saved")

(1349, 3)
[0, 0, 0] missing
                      date  temperature  rainfall
count                 1349       1349.0    1349.0
mean   2024-11-05 00:00:00         12.3       2.7
min    2023-01-01 00:00:00        -22.1       0.0
25%    2023-12-04 00:00:00          4.2       0.0
50%    2024-11-05 00:00:00         13.3       0.0
75%    2025-10-08 00:00:00         21.6       1.8
max    2026-09-10 00:00:00         32.0      90.2
std                    NaN         10.6       6.7
Saved


In [18]:
import holidays

dates = pd.date_range("2023-01-01", "2026-09-10", freq="D")

il_holidays = holidays.country_holidays("US", subdiv="IL", years=range(2023, 2027))

holiday_daily = pd.DataFrame({"date": dates})
holiday_daily["is_holiday"] = holiday_daily["date"].dt.date.isin(il_holidays.keys()).astype(int)
holiday_daily["is_new_year"] = ((holiday_daily["date"].dt.month == 1) &
                                (holiday_daily["date"].dt.day == 1)).astype(int)

print(holiday_daily.shape)                                  # (1349, 3)
print("Holiday days:", holiday_daily["is_holiday"].sum())
print("New Year days:", holiday_daily["is_new_year"].sum())

# show the holidays found, so you can check them
found = holiday_daily[holiday_daily["is_holiday"] == 1]["date"].dt.date
print([(d, il_holidays.get(d)) for d in found][:20])

holiday_daily.to_parquet("../data/processed/holiday_daily.parquet", index=False)
print("Saved")

(1349, 3)
Holiday days: 53
New Year days: 4
[(datetime.date(2023, 1, 1), "New Year's Day"), (datetime.date(2023, 1, 2), "New Year's Day (observed)"), (datetime.date(2023, 1, 16), 'Martin Luther King Jr. Day'), (datetime.date(2023, 2, 12), "Lincoln's Birthday"), (datetime.date(2023, 2, 13), "Lincoln's Birthday (observed)"), (datetime.date(2023, 2, 20), "Washington's Birthday"), (datetime.date(2023, 3, 6), 'Casimir Pulaski Day'), (datetime.date(2023, 5, 29), 'Memorial Day'), (datetime.date(2023, 6, 19), 'Juneteenth National Independence Day'), (datetime.date(2023, 7, 4), 'Independence Day'), (datetime.date(2023, 9, 4), 'Labor Day'), (datetime.date(2023, 10, 9), 'Columbus Day'), (datetime.date(2023, 11, 10), 'Veterans Day (observed)'), (datetime.date(2023, 11, 11), 'Veterans Day'), (datetime.date(2023, 11, 23), 'Thanksgiving Day'), (datetime.date(2023, 12, 25), 'Christmas Day'), (datetime.date(2024, 1, 1), "New Year's Day"), (datetime.date(2024, 1, 15), 'Martin Luther King Jr. Day'), (dat

In [19]:
import pandas as pd

crimes = pd.read_parquet("../data/processed/crimes_clean.parquet")
area_info = pd.read_parquet("../data/processed/area_info.parquet")
weather_daily = pd.read_parquet("../data/processed/weather_daily.parquet")
holiday_daily = pd.read_parquet("../data/processed/holiday_daily.parquet")

n_before = len(crimes)

# a plain calendar date to join weather and holidays on
crimes["date"] = pd.to_datetime(crimes["Date"]).dt.normalize()

# merge, each time checking that no crime row is duplicated
df = crimes.merge(area_info, left_on="Community Area", right_on="community_area",
                  how="left", validate="many_to_one")
df = df.merge(weather_daily, on="date", how="left", validate="many_to_one")
df = df.merge(holiday_daily, on="date", how="left", validate="many_to_one")

print("Rows before:", n_before, "| after:", len(df))       # must be equal
print(df.shape)

new_cols = ["population", "poverty_rate", "unemployment", "per_capita_income",
            "temperature", "rainfall", "is_holiday", "is_new_year"]
print(df[new_cols].isna().sum())                           # all should be 0

Rows before: 922290 | after: 922290
(922290, 33)
population           1277
poverty_rate         1277
unemployment         1277
per_capita_income    1277
temperature             0
rainfall                0
is_holiday              0
is_new_year             0
dtype: int64


In [20]:
bad = df[df["population"].isna()]
print(bad["Community Area"].value_counts().head(10))
print("Valid range in area_info:", area_info["community_area"].min(), "to", area_info["community_area"].max())
print(bad["Date"].dt.year.value_counts().sort_index())

Community Area
0    1277
Name: count, dtype: int64
Valid range in area_info: 1 to 77
Date
2023    367
2024    336
2025    321
2026    253
Name: count, dtype: int64


In [21]:
before = len(df)
df = df[df["Community Area"].between(1, 77)].copy()
print("Removed:", before - len(df), "| rows now:", len(df))

check_cols = ["population", "poverty_rate", "unemployment", "per_capita_income",
              "temperature", "rainfall", "is_holiday", "is_new_year"]
print(df[check_cols].isna().sum().tolist())     # all zeros

df.to_parquet("../data/processed/crimes_enriched.parquet", index=False)
print("Saved", df.shape)

Removed: 1277 | rows now: 921013
[0, 0, 0, 0, 0, 0, 0, 0]
Saved (921013, 33)
